In [2]:
import pandas as pd

df = pd.read_csv("winemag-data-130k-v2.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     129971 non-null  int64  
 1   country                129908 non-null  object 
 2   description            129971 non-null  object 
 3   designation            92506 non-null   object 
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  object 
 7   region_1               108724 non-null  object 
 8   region_2               50511 non-null   object 
 9   taster_name            103727 non-null  object 
 10  taster_twitter_handle  98758 non-null   object 
 11  title                  129971 non-null  object 
 12  variety                129970 non-null  object 
 13  winery                 129971 non-null  object 
dtypes: float64(1), int64(2), object(11)


In [4]:
(len(df) - df.count(0)) / len(df) * 100

id                        0.000000
country                   0.048472
description               0.000000
designation              28.825661
points                    0.000000
price                     6.921544
province                  0.048472
region_1                 16.347493
region_2                 61.136715
taster_name              20.192197
taster_twitter_handle    24.015357
title                     0.000000
variety                   0.000769
winery                    0.000000
dtype: float64

In [5]:
print("mean : " , df['price'].mean())
print("median : " , df['price'].median())
print("variance : " , df['price'].std() ** 2)
print("Standard deviastion : " , df['price'].std())
print("Min : " , df['price'].min())
print("Max : " , df['price'].max())


mean :  35.363389129985535
median :  25.0
variance :  1682.822342407928
Standard deviastion :  41.02221766808723
Min :  4.0
Max :  3300.0


In [6]:
def analyze_column(target_col):
    statistic_data = {
        "mean" : target_col.mean() ,
        "median" : target_col.median(),
        "variance" : target_col.var(),
        "standard_deviation" : target_col.std(),
        "Min" :  target_col.min(),
        "Max"  : target_col.max(),
        "Q1" : target_col.quantile(0.25),
        "Q3" : target_col.quantile(0.75),
        "skewness" : target_col.skew()
    }

    statistic_data = pd.Series(statistic_data)
    return statistic_data
    

analyze_column(df["price"])

mean                    35.363389
median                  25.000000
variance              1682.822342
standard_deviation      41.022218
Min                      4.000000
Max                   3300.000000
Q1                      17.000000
Q3                      42.000000
skewness                18.000957
dtype: float64

In [7]:
def analyze_dataset(df , cols):
    results = pd.DataFrame(columns=cols)

    for col in cols :
        results[col] = analyze_column(df[col])

    return results 

analyze_dataset(df,["price" , "points"])

        

,price,points
mean,35.363389,88.447138
median,25.000000,88.000000
variance,1682.822342,9.239960
standard_deviation,41.022218,3.039730
Min,4.000000,80.000000
Max,3300.000000,100.000000
Q1,17.000000,86.000000
Q3,42.000000,91.000000
skewness,18.000957,0.045921


In [8]:
test_corr = df[['price','points']]
test_corr.corr(method="pearson")

,price,points
price,1.000000,0.416167
points,0.416167,1.000000


In [9]:
def analyze_correlation(df,cols):

    selected_cols = df[cols]
    corr_res = selected_cols.corr(method="pearson")

    return corr_res
analyze_correlation(df,["price","points"])

,price,points
price,1.000000,0.416167
points,0.416167,1.000000


In [16]:
def detect_outliers(col):
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1 

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = col[(col < lower) | (col > upper)]
    
    return {"IQR" : iqr , "Upper_bound" : upper , "Lower_bound":lower ,"outlier_count":outliers.count(),"outlier_percentage":outliers.count() / col.count() * 100 }
    
detect_outliers(df["price"])


{'IQR': np.float64(25.0),
 'Upper_bound': np.float64(79.5),
 'Lower_bound': np.float64(-20.5),
 'outlier_count': np.int64(7241),
 'outlier_percentage': np.float64(5.985534201281256)}

In [17]:
def generate_report(df, cols):
    result = {}

    # 1. Descriptive Statistics
    result["statistics"] = analyze_dataset(df, cols).T

    # 2. Correlation Analysis
    result["correlation"] = analyze_correlation(df, cols)

    # 3. Outlier Detection
    outliers = {}

    for col in cols:
        outliers[col] = detect_outliers(df[col])

    result["outliers"] = pd.DataFrame.from_dict(
        outliers, orient="index"
    )

    return result

In [18]:
report = generate_report(df, ["price", "points"])

In [ ]:
display(report["statistics"])
display(report["correlation"])
display(report["outliers"])

,mean,median,variance,standard_deviation,Min,Max,Q1,Q3,skewness
price,35.363389,25.0,1682.822342,41.022218,4.0,3300.0,17.0,42.0,18.000957
points,88.447138,88.0,9.239960,3.039730,80.0,100.0,86.0,91.0,0.045921


,price,points
price,1.000000,0.416167
points,0.416167,1.000000


,IQR,Upper_bound,Lower_bound,outlier_count,outlier_percentage
price,25.0,79.5,-20.5,7241,5.985534
points,5.0,98.5,78.5,52,0.040009


: 